# Limpieza y normalización de comentarios

Este notebook lee los CSV de `data/etiquetado_humano`, normaliza texto, recupera información semántica de emojis/emoticonos, unifica los archivos indicando la categoría y exporta el archivo limpios  a `data/limpieza_final`. También elimina columnas `Unnamed` y filas cuyo comentario quede vacío después de la limpieza.

In [38]:
#Librerias de limpieza
from __future__ import annotations
import regex as re
import pandas as pd
from pathlib import Path
import os
import unicodedata

try:
    import ftfy
except ImportError:  # portable: funciona sin ftfy
    ftfy = None

In [39]:
#Extracción de rutas para la lectura y derivación de archivos
PROJECT_ROOT = Path(os.getcwd()).parent

# Ruta completa a carpetas
ETIQUETADO_PATH = PROJECT_ROOT / "data" / "etiquetado_humano" 
LIMPIEZA_PATH = PROJECT_ROOT / "data" / "limpieza_final"

In [40]:
def reparar_encoding(texto):
    """Repara texto con mojibake común sin eliminar emojis."""
    if not isinstance(texto, str):
        return ""

    try:
        texto = texto.encode("latin1").decode("utf-8")
    except Exception:
        pass

    texto = ftfy.fix_text(texto)

    reemplazos = {
        "Ã¡": "á", "Ã©": "é", "Ã­": "í", "Ã\xad": "í",
        "Ã³": "ó", "Ã\x93": "Ó", "Ãº": "ú",
        "Ã±": "ñ", "Ã\x91": "Ñ", "Ã\x8d": "Í", "Ã\x81": "Á",
        "Â¿": "¿", "Â¡": "¡", "Â": "", "�": "",
    }

    for mal, bien in reemplazos.items():
        texto = texto.replace(mal, bien)

    return re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", texto)


def fix_hyphenation(texto):
    """Reconecta palabras cortadas por guion al final de línea."""
    if not isinstance(texto, str):
        return ""
    return re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", texto)


def normalize_whitespace(texto):
    """Colapsa espacios, tabs y saltos de línea."""
    if not isinstance(texto, str):
        return ""
    return re.sub(r"\s+", " ", texto).strip()

In [41]:
def leer_csv_robusto(ruta):
    """Lee CSV conservando emojis; usa utf-8-sig y cae a latin-1 si es necesario."""
    try:
        return pd.read_csv(ruta, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(ruta, encoding="latin-1")


def normalizar_columnas(df):
    """Normaliza nombres de columnas y elimina columnas vacías exportadas como Unnamed."""
    df = df.copy()
    df.columns = (
        df.columns
        .str.replace("ï»¿", "", regex=False)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
        .str.lower()
    )
    return df.loc[:, ~df.columns.str.contains(r"^unnamed", case=False, na=False)]


In [42]:
EMOJIS_RISA = {"😂", "🤣", "😆", "😹", "😅", "😁", "😄", "😃"}

EMOJIS_ALEGRIA = {
    "😊", "🙂", "☺️", "😍", "🥰", "😘",
    "😻", "🤩", "😎", "😇", "😋", "😌",
    "❤️", "❤", "💖", "💕", "💜", "💙", "💚"
}

EMOJIS_TRISTEZA = {
    "😢", "😭", "☹️", "🙁", "😞",
    "😔", "😟", "🥲", "💔"
}

EMOJIS_ENOJO = {"😡", "🤬", "😠", "😤", "👿", "💢"}

EMOTICONOS = {
    r"(:\)+|:-\)+|=\)+)": "me gusta",
    r"(:d+|:-d+|xd+)": "me da risa",
    r"(:\(+|:-\(+|=\(+)": "me entristece",
    r"(>:\(+|d:<)": "me enoja",
}
MEXICANISMOS = {
    r"\bnmms\b": "no me gusta",
    r"\bno mames\b": "sorprendente",
    r"\bno manches\b": "sorprendente",
    r"\bchido\b": "me gusta",
    r"\bpadre\b": "me gusta",
    r"\bgenial\b": "me gusta",
    r"\bculero\b": "malo",
    r"\bculerisimo\b": "muy malo",
    r"\bgacho\b": "malo",
    r"\bpinche\b": "",
    r"\bverga\b": "",
    r"\bchingon\b": "muy bueno",
    r"\bchingona\b": "muy bueno",
    r"\bchingones\b": "muy bueno",
    r"\bchingar\b": "molestar",
    r"\bchingada\b": "muy malo",
    r"\bvale madre\b": "malo",
    r"\bvalio madre\b": "muy malo",
    r"\bqlero\b": "malo",
    r"\bogt\b": "persona desagradable",
    r"\bpoca madre\b": "muy bueno",
    r"\ba toda madre\b": "muy bueno",
}


def intensidad(contador, frase):
    if contador == 0:
        return ""
    if contador == 1:
        return frase
    if contador <= 3:
        return f"{frase} mucho"
    return f"{frase} muchisimo"


def reemplazar_emojis_repetidos(texto, emojis, frase):
    contador = 0
    for emoji in emojis:
        ocurrencias = texto.count(emoji)
        if ocurrencias:
            contador += ocurrencias
            texto = texto.replace(emoji, " ")

    intensidad_texto = intensidad(contador, frase)
    if intensidad_texto:
        texto += f" {intensidad_texto} "
    return texto


def limpiar_texto_es(texto):
    if not isinstance(texto, str):
        return ""

    texto = reparar_encoding(texto)
    texto = unicodedata.normalize("NFKC", texto.lower())

    texto = re.sub(r"\[sticker\]", " ", texto, flags=re.IGNORECASE)
    texto = re.sub(r"https?://\S+|www\.\S+", " ", texto)
    texto = re.sub(r"\bx{4,}\b"," me da risa ",texto,flags=re.IGNORECASE)

    for patron, reemplazo in EMOTICONOS.items():
        texto = re.sub(patron, f" {reemplazo} ", texto, flags=re.IGNORECASE)

    patrones_risa = [r"\b(ja){2,}\b", r"\b(ha){2,}\b", r"\bjeje+\b", r"\bjiji+\b", r"\bxd+\b"]
    for patron in patrones_risa:
        texto = re.sub(patron, " me da risa ", texto, flags=re.IGNORECASE)

    for patron, reemplazo in MEXICANISMOS.items():
        texto = re.sub(patron,f" {reemplazo} ",texto,flags=re.IGNORECASE)
    

    texto = reemplazar_emojis_repetidos(texto, EMOJIS_RISA, "me da risa")
    texto = reemplazar_emojis_repetidos(texto, EMOJIS_ALEGRIA, "me gusta")
    texto = reemplazar_emojis_repetidos(texto, EMOJIS_TRISTEZA, "me entristece")
    texto = reemplazar_emojis_repetidos(texto, EMOJIS_ENOJO, "me enoja")

    texto = re.sub(r"\p{Emoji_Presentation}|\p{Extended_Pictographic}", " ", texto)

    replacements = {
        "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
        "\u2013": "-", "\u2014": "-", "\u2026": " ",
        "\u00a0": " ", "\ufeff": "", "\u200b": "",
    }
    for old, new in replacements.items():
        texto = texto.replace(old, new)

    texto = re.sub(r"[\x00-\x1f\x7f]", " ", texto)
    texto = re.sub(r"[^a-záéíóúüñ0-9\s]", " ", texto)
    return normalize_whitespace(texto)


In [43]:
def limpiar_dataframe(df, columna_texto="comentario"):
    """Limpia comentarios y elimina filas vacías antes y después de la limpieza."""
    df = normalizar_columnas(df)

    if columna_texto not in df.columns:
        raise ValueError(f"No existe la columna '{columna_texto}'. Columnas disponibles: {df.columns.tolist()}")

    n_original = len(df)

    df = df.dropna(subset=[columna_texto]).copy()
    df[columna_texto] = (
        df[columna_texto]
        .astype(str)
        .apply(reparar_encoding)
        .apply(fix_hyphenation)
        .apply(normalize_whitespace)
        .apply(limpiar_texto_es)
    )

    # Elimina filas cuyo comentario quedó vacío después de quitar stickers, URLs, emojis no mapeados o símbolos.
    df = df[df[columna_texto].str.strip().ne("")].copy()

    return df, n_original - len(df)


In [44]:
archivos_entrada = [
    "infraestructura_etiquetado_humano",
    "turismo_etiquetado_humano",
    "seguridad_etiquetado_humano",
]

archivos_salida = ["infraestructura", "turismo", "seguridad"]

for entrada, salida in zip(archivos_entrada, archivos_salida):
    ruta_entrada = ETIQUETADO_PATH / f"{entrada}.csv"
    ruta_salida = LIMPIEZA_PATH / f"{salida}_limpio.csv"

    datos = leer_csv_robusto(ruta_entrada)
    datos_limpios, eliminadas = limpiar_dataframe(datos, columna_texto="comentario")

    datos_limpios.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

    print(f"{salida}: {len(datos_limpios)} filas exportadas | {eliminadas} filas vacías eliminadas")

print("Limpieza de comentarios terminada")

infraestructura: 825 filas exportadas | 88 filas vacías eliminadas
turismo: 782 filas exportadas | 111 filas vacías eliminadas
seguridad: 971 filas exportadas | 25 filas vacías eliminadas
Limpieza de comentarios terminada


In [45]:
#Unificación de documentos 

LIMPIO_PATH = Path("../data/limpieza_final")

archivos = {
    "infraestructura": LIMPIO_PATH / "infraestructura_limpio.csv",
    "seguridad": LIMPIO_PATH / "seguridad_limpio.csv",
    "turismo": LIMPIO_PATH  / "turismo_limpio.csv",
}

dfs = []

for categoria, ruta in archivos.items():
    df = pd.read_csv(ruta, encoding="utf-8-sig")

    # Eliminar columnas vacías tipo Unnamed
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    # Normalizar nombres de columnas
    df.columns = df.columns.str.replace("\ufeff", "", regex=False).str.strip()

    # Agregar categoría
    df["categoria"] = categoria

    dfs.append(df)

df_unificado = pd.concat(dfs, ignore_index=True)

df_unificado.to_csv(
    LIMPIO_PATH / "etiquetado_humano_unificado.csv",
    index=False,
    encoding="utf-8-sig"
)

df_unificado.head()

,comentario,etiquetado_humano,categoria
0,cuál es el más cercado para rayar el nombre de...,3.0,infraestructura
1,esos baños deberian estar en el metro no saben...,2.0,infraestructura
2,no pues bueno me da risa,3.0,infraestructura
3,los van a abandonar,2.0,infraestructura
4,nada los tiene contentos,4.0,infraestructura
